# 02d: physical-scale (mm) cache, one shard

Same pipeline as 02b with geometry "mm": 140 mm per 256 px, crop centred on the largest
body region, 32 x 256 x 256 uint8 per plane. Split into two shards because the full mm
cache (about 19 GB) is too close to the 20 GB output limit.

Run twice: SHARD = 0 and SHARD = 1 (Copy & Edit, change only SHARD). Both outputs are
attached together in training; the cache folder in each is named cache_mm.
Settings: CPU, Internet off. Inputs: competition data, utility script knee_prep (with leg_mask).

# Cell 1: scan, select, this shard

In [ ]:
import os, sys, glob, time, json, datetime
from pathlib import Path
import pandas as pd

SHARD, NSHARDS = 0, 2
GEOMETRY = "mm"
LIMIT = None
ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT = Path("/kaggle/working")

def find_all(pattern):
    hits = glob.glob(f"/kaggle/usr/lib/**/{pattern}", recursive=True)
    for top in os.listdir("/kaggle/input"):
        if top != "competitions":
            hits += glob.glob(f"/kaggle/input/{top}/**/{pattern}", recursive=True)
    return sorted(hits)

sys.path.insert(0, os.path.dirname(find_all("knee_prep.py")[0]))
import knee_prep as kp
print("MM_FOV", kp.MM_FOV, "| leg_mask:", hasattr(kp, "leg_mask"))

t0 = time.time()
ser = pd.read_csv(f"{ROOT}/train_series.csv")
inv = kp.scan_all(ser, ROOT, "train_series")
sel = kp.select_series(inv)
ids = sorted(sel["StudyInstanceUID"].unique())
mine = ids[SHARD::NSHARDS]
if LIMIT:
    mine = mine[:LIMIT]
sel = sel[sel["StudyInstanceUID"].isin(mine)]
print(f"scan {time.time() - t0:.0f}s | shard {SHARD}: {len(mine)} studies, {len(sel)} series")

# Cell 2: build

In [ ]:
t0 = time.time()
qa = kp.build_cache(sel, ROOT, "train_series", str(OUT / "cache_mm"), workers=4, geometry=GEOMETRY)
files = list((OUT / "cache_mm").glob("*.npz"))
gb = sum(f.stat().st_size for f in files) / 1e9
print(f"built {len(files)} of {len(mine)} studies in {(time.time() - t0) / 60:.1f} min, {gb:.2f} GB")
print("errors:", int(qa["error"].notna().sum()) if len(qa) else 0)
if len(qa):
    print(qa[["dup_pos", "big_gaps", "mixed_shapes"]].sum().to_string())
    print("series without pixel spacing (fell back to px):", int(qa["px"].isna().sum()))
qa.to_csv(OUT / f"cache_mm_qa_shard{SHARD}.csv", index=False)

# Cell 3: lineage

In [ ]:
meta = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "source": "rsna-knee-abnormality-detection/train_series",
    "geometry": GEOMETRY, "mm_fov": kp.MM_FOV, "shape": [kp.D, kp.S, kp.S], "dtype": "uint8",
    "shard": SHARD, "nshards": NSHARDS, "studies": len(mine),
    "crop": "centred on the largest connected body region of the mean slice",
    "git_commit": "FILL_ME",
}
json.dump(meta, open(OUT / f"cache_mm_meta_shard{SHARD}.json", "w"), indent=2)
print(json.dumps(meta, indent=1))